# Exercise 7: Deploying a Keras Model to Azure ML
**Module 2 — Deep Learning and Neural Networks**

In this notebook you will:
1. Train a simple Keras binary classifier
2. Save and register the model in Azure ML
3. Write a scoring script for inference
4. Deploy the model as a REST endpoint on Azure Container Instance (ACI)
5. Test the live endpoint
6. Clean up resources

**Prerequisites:**
- An Azure ML Workspace (`ml-workshop-ws`) already created
- `config.json` downloaded from Azure ML Studio and uploaded to this Colab session

**Estimated time:** 60 minutes (10 min setup, 10 min train/register, 15 min deploy, 10 min test, 5 min cleanup)


## Section 0: Install Dependencies

Run this cell once. It takes 2-4 minutes.

In [ ]:
# Install the Azure ML SDK
# This does not affect other Colab exercises — packages are session-scoped
!pip install azureml-core azureml-sdk --quiet
print("Installation complete")

## Section 1: Connect to Azure ML Workspace

A **Workspace** is the top-level resource in Azure ML. It holds your registered models,
experiments, compute targets, and deployed endpoints.

Make sure you have uploaded `config.json` to this Colab session before running the next cell.
You can upload files using the folder icon in the left panel.


In [ ]:
from azureml.core import Workspace

# Reads subscription_id, resource_group, and workspace_name from config.json
ws = Workspace.from_config()

print(f"Connected to workspace : {ws.name}")
print(f"Resource group        : {ws.resource_group}")
print(f"Azure region          : {ws.location}")

**What just happened?**
`Workspace.from_config()` authenticated with Azure (you may have completed a browser login step)
and returned a Python object that acts as your handle to everything in the workspace.
All subsequent SDK calls pass `ws` as their target.


## Section 2: Train and Save the Model

We will train a small feedforward network on synthetic heart disease data — the same architecture
from Exercise 1. In a real project you would load your actual trained model from a previous session.

The key requirement: the model must be saved to disk before registration.


In [ ]:
import tensorflow as tf
from tensorflow import keras
import numpy as np
import os

# ── Synthetic dataset (replace with your actual data) ─────────────────────────
np.random.seed(42)
X_train = np.random.rand(300, 13).astype("float32")
y_train = (np.random.rand(300) > 0.5).astype("float32")

# ── Model architecture: matches Exercise 1 ────────────────────────────────────
model = keras.Sequential([
    keras.Input(shape=(13,)),
    keras.layers.Dense(16, activation="relu"),
    keras.layers.Dense(8,  activation="relu"),
    keras.layers.Dense(1,  activation="sigmoid")
], name="heart_disease_classifier")

model.compile(
    optimizer = keras.optimizers.Adam(learning_rate=0.001),
    loss      = "binary_crossentropy",
    metrics   = ["accuracy"]
)

print(model.summary())

In [ ]:
# Train the model
history = model.fit(
    X_train, y_train,
    epochs          = 20,
    batch_size      = 32,
    validation_split= 0.2,
    verbose         = 1
)

final_acc = history.history["val_accuracy"][-1]
print(f"\nFinal validation accuracy: {final_acc:.3f}")

In [ ]:
# Save the model
# Azure ML convention: write outputs to the outputs/ folder
os.makedirs("outputs", exist_ok=True)
model.save("outputs/heart_model.h5")

print("Model saved to: outputs/heart_model.h5")
print(f"File size     : {os.path.getsize('outputs/heart_model.h5') / 1024:.1f} KB")

**Why `outputs/`?**
When training runs on Azure ML compute, everything written to `outputs/` is automatically
uploaded to the workspace as a run artifact. This habit keeps your local and cloud workflows consistent.


## Section 3: Register the Model

Registering uploads the model file to the workspace blob storage and assigns it a
**name** and **version number**. Every re-registration of the same name auto-increments the version.

This version history lets you roll back to a previous model if a new deployment underperforms.


In [ ]:
from azureml.core.model import Model

registered_model = Model.register(
    workspace   = ws,
    model_path  = "outputs/heart_model.h5",      # local file to upload
    model_name  = "heart-disease-classifier",    # name in the registry
    description = "Binary classifier: heart disease from 13 clinical features",
    tags        = {
        "framework"   : "keras",
        "task"        : "binary-classification",
        "exercise"    : "module2-ex7"
    }
)

print(f"Registered model : {registered_model.name}")
print(f"Version          : {registered_model.version}")
print(f"ID               : {registered_model.id}")

**Verify in Azure ML Studio:**
Go to ml.azure.com → your workspace → **Models** in the left panel.
You should see `heart-disease-classifier` listed with version 1.


## Section 4: Write the Scoring Script

The scoring script is the code that runs **inside the deployed container**.
It has two required functions:

| Function | When called | Responsibility |
|---|---|---|
| `init()` | Once at container startup | Load model into memory |
| `run(raw_data)` | On every prediction request | Parse input → predict → return JSON |

**Critical rule:** Load the model in `init()`, not in `run()`. Loading in `run()` means
the model is reloaded from disk on every request, adding 3-8 seconds of latency per call.


In [ ]:
%%writefile score.py
import json
import numpy as np
import tensorflow as tf
from azureml.core.model import Model


def init():
    """
    Runs once when the container starts.
    Loads the registered model into a module-level variable.
    """
    global model

    # Model.get_model_path finds the file by the registered name
    model_path = Model.get_model_path("heart-disease-classifier")
    model = tf.keras.models.load_model(model_path)
    print(f"Model loaded from: {model_path}")


def run(raw_data):
    """
    Runs on every prediction request.

    Expected input (JSON string):
        {"data": [[f1, f2, ..., f13], [f1, f2, ..., f13], ...]}

    Returns (JSON string):
        {"predictions": [0.73, 0.21, ...]}
        where each value is a sigmoid probability (>=0.5 indicates heart disease)
    """
    try:
        payload   = json.loads(raw_data)
        features  = np.array(payload["data"], dtype=np.float32)

        # Validate input shape
        if features.ndim != 2 or features.shape[1] != 13:
            return json.dumps({
                "error": f"Expected shape (n, 13), got {features.shape}"
            })

        raw_preds   = model.predict(features)
        predictions = raw_preds.flatten().tolist()

        return json.dumps({"predictions": predictions})

    except KeyError:
        return json.dumps({"error": "Input JSON must have key 'data'"})
    except Exception as e:
        return json.dumps({"error": str(e)})


In [ ]:
# Verify the file was written correctly
with open("score.py") as f:
    print(f.read())

## Section 5: Define the Inference Environment

The environment specifies exactly which Python packages the container needs at **inference time**.

Important: only include packages needed to run `score.py`. Do not include training-only
packages (matplotlib, seaborn, scikit-learn) — they bloat the image and slow deployment.

`azureml-defaults` is always required; it provides the runtime hooks that connect
your scoring script to the Azure serving infrastructure.


In [ ]:
from azureml.core import Environment
from azureml.core.conda_dependencies import CondaDependencies

# Create a named environment
env = Environment(name="heart-disease-inference-env")

# Build the dependency list
deps = CondaDependencies()
deps.add_pip_package("tensorflow==2.10.0")
deps.add_pip_package("numpy==1.23.5")
deps.add_pip_package("azureml-defaults")   # required — do not remove

env.python.conda_dependencies = deps

print("Environment defined")
print("Packages:")
for pkg in deps.pip_packages:
    print(f"  - {pkg}")

## Section 6: Create Inference Configuration

`InferenceConfig` bundles the scoring script and the environment into a single deployable object.
This is what gets packaged into the Docker container.


In [ ]:
from azureml.core.model import InferenceConfig

inference_config = InferenceConfig(
    entry_script = "score.py",
    environment  = env
)

print("Inference configuration created")
print(f"  Entry script : score.py")
print(f"  Environment  : {env.name}")

## Section 7: Deploy to Azure Container Instance (ACI)

**Azure Container Instance** is a lightweight, serverless container runtime.
It is appropriate for:
- Testing and validation
- Low-traffic APIs (up to ~100 requests per minute)
- Short-lived endpoints

For production with high traffic, you would deploy to **Azure Kubernetes Service (AKS)** instead.

Deployment takes **5-10 minutes**. The SDK polls Azure and prints status updates.


In [ ]:
from azureml.core.webservice import AciWebservice
from azureml.core.model import Model

# Configure the ACI container
aci_config = AciWebservice.deploy_configuration(
    cpu_cores   = 1,       # 1 vCPU is sufficient for this model
    memory_gb   = 1,       # 1GB RAM
    description = "Heart disease prediction REST API — Module 2 Exercise 7"
)

# Deploy
service = Model.deploy(
    workspace         = ws,
    name              = "heart-disease-api",    # endpoint name (lowercase, hyphens only)
    models            = [registered_model],
    inference_config  = inference_config,
    deployment_config = aci_config,
    overwrite         = True                    # overwrite if name already exists
)

# Block until deployment completes (or fails)
service.wait_for_deployment(show_output=True)

print("\n" + "="*50)
print(f"Deployment state : {service.state}")
print(f"Endpoint URL     : {service.scoring_uri}")

In [ ]:
# If state is not Healthy, print the container logs to diagnose
if service.state != "Healthy":
    print("Deployment failed. Container logs:")
    print(service.get_logs())
else:
    print("Service is healthy and ready to accept requests")

## Section 8: Test the Endpoint

We will test the endpoint two ways:
1. Using the `requests` library (standard REST call)
2. Using the Azure ML SDK's built-in `service.run()` shortcut


In [ ]:
import json
import requests

# ── Test payload: 3 patient records, 13 features each ─────────────────────────
test_payload = json.dumps({
    "data": [
        [63, 1, 3, 145, 233, 1, 0, 150, 0, 2.3, 0, 0, 1],   # record 1
        [37, 1, 2, 130, 250, 0, 1, 187, 0, 3.5, 0, 0, 2],   # record 2
        [41, 0, 1, 130, 204, 0, 0, 172, 0, 1.4, 2, 0, 2]    # record 3
    ]
})

headers  = {"Content-Type": "application/json"}
response = requests.post(service.scoring_uri, data=test_payload, headers=headers)

print(f"HTTP status : {response.status_code}")
print(f"Response    : {response.json()}")
print()

# Interpret results
result = response.json()
if "predictions" in result:
    for i, p in enumerate(result["predictions"]):
        label = "Heart disease likely" if p >= 0.5 else "No heart disease"
        print(f"  Record {i+1}: probability = {p:.3f}  →  {label}")

In [ ]:
# ── Alternative: SDK shortcut ─────────────────────────────────────────────────
# service.run() wraps the HTTP call for quick interactive testing
sdk_response = service.run(input_data=test_payload)
print("SDK response:", sdk_response)

In [ ]:
# ── Test error handling: malformed input ──────────────────────────────────────
bad_payload = json.dumps({"wrong_key": [[1, 2, 3]]})
bad_response = requests.post(service.scoring_uri, data=bad_payload, headers=headers)
print("Error handling test:")
print(f"  Status : {bad_response.status_code}")
print(f"  Body   : {bad_response.json()}")

## Section 9: View Deployment Logs

Logs are useful for debugging runtime errors. They capture everything printed to stdout
inside `init()` and `run()`.


In [ ]:
# Print the last 50 lines of container logs
logs = service.get_logs()
lines = logs.split("\n")
print(f"Total log lines: {len(lines)}")
print("\n--- Last 30 lines ---")
print("\n".join(lines[-30:]))

## Section 10: Clean Up

**Always delete ACI services after testing.** A running endpoint charges by the hour,
even if it receives zero requests.

Run the cleanup cell below before closing this notebook.


In [ ]:
# Delete the running service
print("Deleting service...")
service.delete()
print(f"Service '{service.name}' deleted")

# Optional: remove the registered model too
# registered_model.delete()
# print("Model registration removed")

# Verify deletion
try:
    from azureml.core.webservice import Webservice
    svc = Webservice(ws, "heart-disease-api")
    print(f"Service still exists: {svc.state}")
except Exception:
    print("Confirmed: service no longer exists in workspace")

## What's Next?

You have completed the full deployment pipeline. Here are natural next steps:

**For this module:**
- Swap in your LSTM or CNN model from earlier exercises and redeploy
- Add authentication to the endpoint using Azure ML key-based auth
- Monitor requests using Application Insights

**For production deployments:**
- Replace ACI with AKS for auto-scaling under load
- Create a CI/CD pipeline that auto-registers and deploys when model accuracy improves
- Add input validation and output logging to the scoring script

**Challenge task:**
Modify `score.py` to also return the binary class label (`0` or `1`) alongside the probability,
and update your test payload to verify the new output format.
